In [0]:

dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
## PARAMETRIZAR ADLS Y CATALOGO A PROD
dbutils.widgets.text("PRM_catalogo","catalogo_desa_intEcommerce")
dbutils.widgets.text("PRM_nameStorage","adlsecompag")
###########################################
dbutils.widgets.text("PRM_esquema", "bronze")
dbutils.widgets.text("PRM_container","raw-insumos")


In [0]:

PRM_container = dbutils.widgets.get("PRM_container")
PRM_nameStorage = dbutils.widgets.get("PRM_nameStorage")
PRM_catalogo = dbutils.widgets.get("PRM_catalogo")
PRM_esquema = dbutils.widgets.get("PRM_esquema")


rutaINT = f"abfss://{PRM_container}@{PRM_nameStorage}.dfs.core.windows.net/Interaccion_Sistema.csv"
rutaCLI = f"abfss://{PRM_container}@{PRM_nameStorage}.dfs.core.windows.net/Clientes_Sistema.csv"
rutaPRO = f"abfss://{PRM_container}@{PRM_nameStorage}.dfs.core.windows.net/Productos_Sistema.csv"

In [0]:
df_raw_tipointeraccion = spark.read \
                        .format("csv") \
                        .option("sep", ";") \
                        .option("header", "true") \
                        .option("inferSchema", "true") \
                        .load(rutaINT)

df_raw_clientes       =  spark.read \
                        .format("csv") \
                        .option("sep", ";") \
                        .option("header", "true") \
                        .option("inferSchema", "true") \
                        .load(rutaCLI)


df_raw_productos      = spark.read \
                        .format("csv") \
                        .option("sep", ";") \
                        .option("header", "true") \
                        .option("inferSchema", "true") \
                        .load(rutaPRO)




In [0]:
df_raw_tipointeraccion_final = df_raw_tipointeraccion.select(\
                        col("TypeInt_id").alias("TypeInt_id"),	
                        col("Interaction_name").alias("Interaction_name"),	
                        col("Des_Spanish").alias("DestipoInt_Spanish"),	
                        col("System_date").alias("Registration_system_date")                       
).withColumn("Fecha_proceso",to_timestamp(date_format(current_timestamp(), "yyyy-MM-dd")))

df_raw_tipointeraccion_final.display()

In [0]:
df_raw_clientes_final   = df_raw_clientes.select(\
                        col("User_id").alias("User_id"),	
                        col("Last_Name").alias("Last_Name"),	
                        col("Name").alias("Name"),	
                        col("Age").alias("Age"),
                        col("Cell_number").alias("Cell_number"), 
                        col("Sign_date").alias("Sign_date")                 
).withColumn("Fecha_proceso",to_timestamp(date_format(current_timestamp(), "yyyy-MM-dd")))


In [0]:
df_raw_productos_final   = df_raw_productos.select(\
                        col("Product_id").alias("Product_id"),	
                        col("Producto_Name").alias("Producto_Name"),	
                        col("Category").alias("Category"),	
                        col("Price_PEN").alias("Price")              
).withColumn("Fecha_proceso",to_timestamp(date_format(current_timestamp(), "yyyy-MM-dd")))


In [0]:
df_raw_tipointeraccion_final.write \
    .mode("overwrite") \
    .saveAsTable(f"{PRM_catalogo}.{PRM_esquema}.Interaccion_Sistema")

df_raw_clientes_final.write \
    .mode("overwrite") \
    .saveAsTable(f"{PRM_catalogo}.{PRM_esquema}.Clientes_Sistema")

df_raw_productos_final.write \
    .mode("overwrite") \
    .saveAsTable(f"{PRM_catalogo}.{PRM_esquema}.Productos_Sistema")

